# 06 · The evaluation set
Loads the 50-question eval set, checks that every evidence phrase exists in a chunk of its
expected article, and shows the questions for review.

In [1]:
# ===== CELL 1: Load the eval set and the chunks =====
# The eval set holds the questions and their known answers; the chunks are what retrieval searches.

import re                       # re: text patterns (used to tidy whitespace)
import pandas as pd             # pandas: tables

evals = pd.read_json("../evals/eval_set.jsonl", lines=True)          # Load the 50 questions
chunks = pd.read_json("../data/processed/chunks.jsonl", lines=True)  # Load the 6,601 chunks

print("Questions:", len(evals))                                       # Should print 50
print(pd.crosstab(evals["area"], evals["type"]))                     # Table: how many questions of each type per area

Questions: 50
type          direct  out_of_scope  paraphrase
area                                          
blazor            10             0           5
fundamentals       9             0           6
mvc               11             0           4
out_of_scope       0             5           0


In [2]:
# ===== CELL 2: Validate the evidence phrases =====
# For each question, checks that its evidence phrase appears in at least one chunk
# of the expected article. If not, retrieval could never "find" it, so the question would be unfair.

def norm(text):                                         # Make text easy to compare...
    return re.sub(r"\s+", " ", text).lower()            # ...by collapsing all whitespace to single spaces and lowercasing

chunks["norm"] = chunks["text"].map(norm)               # Pre-compute the normalised text of every chunk (faster than doing it per question)

results = []                                            # One row per in-scope question goes here
for q in evals[evals["doc_id"].notna()].itertuples():   # Loop over questions that have an expected article (skips the 5 out-of-scope ones)
    in_doc = chunks[chunks["doc_id"] == q.doc_id]       # All chunks from the expected article
    matches = in_doc[in_doc["norm"].str.contains(norm(q.evidence), regex=False)]   # Those containing the evidence phrase (regex=False: treat it as plain text)
    results.append({                                    # Record the result
        "id": q.id,                                     # Question ID, e.g. F02
        "doc_chunks": len(in_doc),                      # How many chunks the article has
        "evidence_chunks": len(matches),                # How many of them contain the evidence
        "chunk_ids": ", ".join(matches["chunk_id"]),    # Which ones (these are the "correct answers" for retrieval)
    })

check = pd.DataFrame(results)                           # Turn the results into a table
print("Questions where the evidence was NOT found:", (check["evidence_chunks"] == 0).sum())   # Should print 0
check                                                   # Show the full table

Questions where the evidence was NOT found: 0


,id,doc_chunks,evidence_chunks,chunk_ids
0,F01,41,1,fundamentals/dependency-injection::5
1,F02,41,1,fundamentals/dependency-injection::5
2,F03,41,1,fundamentals/dependency-injection::14
3,F04,41,2,"fundamentals/dependency-injection::15, fundame..."
4,F05,40,1,fundamentals/middleware/index::21
5,F06,40,1,fundamentals/middleware/index::0
6,F07,41,1,fundamentals/configuration/options::9
7,F08,41,1,fundamentals/configuration/options::8
8,F09,26,1,fundamentals/environments::4
9,F10,55,1,fundamentals/static-files::0


In [3]:
# ===== CELL 3: Review the questions and expected answers =====
# Prints every question with its expected answer, so you can check them as the .NET expert.

for q in evals.itertuples():                            # Loop over all 50 questions
    print(f"[{q.id}] ({q.type}) {q.question}")          # ID, type and question
    print(f"      → {q.expected_answer}\n")             # The expected answer, then a blank line

[F01] (direct) Which method registers a service with a scoped lifetime, and how long does a scoped service live in an MVC app?
      → Use AddScoped. A scoped service lives for a single request in an MVC or Razor Pages app (for a Blazor app, the lifetime of the circuit).

[F02] (paraphrase) How do I register a service so a new instance is created per HTTP request?
      → Register it with AddScoped. A scoped lifetime is a single request in an MVC or Razor Pages app.

[F03] (paraphrase) My custom middleware throws a runtime exception when I inject a scoped service into its constructor. What should I do instead?
      → Constructor injection forces the scoped service to behave like a singleton. Inject the service into the middleware's Invoke or InvokeAsync method instead, or use factory-based middleware, which is activated per request.

[F04] (direct) How do I register and resolve keyed services?
      → Register them with a key using AddKeyedSingleton, AddKeyedScoped or AddKeyedTransien